# 06 Tabular Transformer (PyTorch) - House Price Prediction

This is a complete, standalone pipeline for training a Tabular Transformer using PyTorch on dirty house price data.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score
import re

sns.set(style='whitegrid')

### 1. Preprocessing & Embedding Preparation

In [ ]:
def clean_house_data(df):
    df = df.copy().dropna(subset=['Price']).drop_duplicates()
    def parse_lot(v):
        if pd.isna(v): return 10000
        v = str(v).lower()
        num = float(re.findall(r'\d+\.\d+|\d+', v)[0])
        if 'ac' in v: return num * 43560
        return num
    df['Lot_Size'] = df['Lot_Size'].apply(parse_lot)
    df['Bedrooms'] = df['Bedrooms'].apply(lambda x: sum([int(i) for i in str(x).split('+')]) if '+' in str(x) else int(re.sub(r'\D', '', str(x))))
    df['Price'] = df['Price'].clip(upper=df['Price'].quantile(0.95))
    return df

df = clean_house_data(pd.read_csv('../_data/house_prices.csv'))

cat_cols = ['Neighborhood', 'House_Style', 'Property_Type']
num_cols = ['Living_Area', 'Lot_Size', 'Bedrooms', 'Bathrooms', 'Garage_Capacity']

for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

X = df[cat_cols + num_cols]
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. Tabular Transformer Model

In [ ]:
class TabTransformer(nn.Module):
    def __init__(self, cat_dims, num_len, embed_dim=16):
        super().__init__()
        self.cat_embeddings = nn.ModuleList([nn.Embedding(dim, embed_dim) for dim in cat_dims])
        self.num_projections = nn.ModuleList([nn.Linear(1, embed_dim) for _ in range(num_len)])
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.head = nn.Sequential(nn.Linear(embed_dim * (len(cat_dims) + num_len), 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x_cat, x_num):
        c_emb = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)]
        n_emb = [proj(x_num[:, i].unsqueeze(-1)) for i, proj in enumerate(self.num_projections)]
        x = torch.stack(c_emb + n_emb, dim=1)
        x = self.transformer(x).flatten(1)
        return self.head(x)

cat_dims = [df[c].nunique() for c in cat_cols]
model = TabTransformer(cat_dims, len(num_cols)).to(torch.device('cpu'))
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

### 3. Training Loop

In [ ]:
X_train_cat = torch.tensor(X_train[cat_cols].values, dtype=torch.long)
X_train_num = torch.tensor(X_train[num_cols].values, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

model.train()
for epoch in range(50):
    optimizer.zero_grad()
    preds = model(X_train_cat, X_train_num)
    loss = criterion(preds, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0: print(f"Epoch {epoch+1}, Loss: {loss.item():.2f}")

### 4. Evaluation & Visuals

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(torch.tensor(X_test[cat_cols].values, dtype=torch.long), 
                   torch.tensor(X_test[num_cols].values, dtype=torch.float32)).numpy()

print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
plt.figure(figsize=(10, 6))
sns.residplot(x=y_test, y=y_pred.flatten(), color='orange')
plt.title('Residual Plot - TabTransformer')
plt.show()

### 5. Model Saving

In [ ]:
torch.save(model.state_dict(), '../_model/house_prediction_tab_transformer.pth')
print("TabTransformer Model saved.")